# README

- **Author**: RUEI-HSIANG, CHANG（張睿庠，B12303030；AI協助）
- **Created At**: 2026-11-11
- **Last Modified At**: 2026-11-11

## What does this file do?

依使用者本次更正，分群改用G1–G6，取代先前只用G5/G6的分析範圍。使用者明確選擇完整觀察樣本：210人中21人缺年齡，全部為議員且G5=1；保留189人（11首長、178議員），未補值。原G5/G6 finding結果已依使用者要求移除；來源CSV保留歷史資料快照。

建立六個有意義的分析欄：G1年齡為連續；G2性別、G3三類公開學歷、G4五類登記政黨為類別；G5與G6為對稱二元。G3沿用既有政策碼，剩5筆未公開學歷低類別旗標；不冒稱學位已認證。G6=0為記錄搜尋未發現符合範圍親屬。游智彬研究暫納入與資格限制保留。未更改原背景值。

## What does this file take?

- `data/processed/tk_g5_g6_characteristics_20261110.csv`；SHA-256 `9df3c70602e5ac1293422dfed1a4a008e07d7280b069225d30a3cb250000a8a0`；[上游文件](../processed/tk_g5_g6_characteristics_20261110_documentation.ipynb)，原210人研究名冊與背景，不重查2026-11-11官方資格。

## What does this file output?

- `data/processed/tk_g1_g6_complete_20261111.csv`：189人完整G1–G6分析輸入，candidate_key唯一鍵。
- `data/processed/tk_g1_g6_exclusions_20261111.csv`：21人排除清單，保留姓名、城市、G5與缺欄理由。
- 兩檔均由[整理後文件](../processed/tk_g1_g6_20261111_documentation.ipynb)綁定；實際筆數、schema及hash由下方磁碟回讀驗證。


In [1]:
from pathlib import Path
import hashlib,json,re
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
for ROOT in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
    if (ROOT/'AGENTS.md').is_file() and (ROOT/'templates').is_dir(): break
else: raise FileNotFoundError('請從專案目錄執行')
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def checked(rel,h):
    p=(ROOT/rel).resolve()
    assert p.is_relative_to((ROOT/'data/processed').resolve()) and p.is_file(),rel
    assert sha(p)==h,f'輸入版本改變：{rel}'
    return p
def save(frame,rel):
    p=(ROOT/rel).resolve(); assert p.is_relative_to((ROOT/'data/processed').resolve())
    frame.to_csv(p,index=False,encoding='utf-8-sig')
    q=pd.read_csv(p,dtype=str,keep_default_na=False)
    pd.testing.assert_frame_equal(frame.fillna('').astype(str).reset_index(drop=True),q,check_dtype=False)
    print(rel,len(q),len(q.columns),sha(p))


In [2]:
p=checked('data/processed/tk_g5_g6_characteristics_20261110.csv','9df3c70602e5ac1293422dfed1a4a008e07d7280b069225d30a3cb250000a8a0')
d=pd.read_csv(p,dtype=str,keep_default_na=False)
COLS={'G1_age':'G1_age_2026_minus_birth_year','G2_sex_category':'G2_sex','G3_education_category':'G3_education_level','G4_party_category':'G4_party','G5_first_candidacy':'G5_first_time_any_elected_office','G6_family_search':'G6_political_family'}
assert len(d)==210 and d.candidate_key.is_unique
missing=d[list(COLS.values())].eq('')
keep=~missing.any(axis=1)
x=d.loc[keep].copy()
for new,old in COLS.items(): x[new]=x[old]
x['G1_age']=pd.to_numeric(x.G1_age,errors='raise').astype(int)
x=x.sort_values('candidate_key',kind='stable').reset_index(drop=True)
e=d.loc[~keep,['candidate_key','candidate_name','county_city','office_type','G5_first_time_any_elected_office','G3_unpublished_lowest_flag']].copy()
e['missing_fields']=[';'.join(missing.columns[missing.loc[i]]) for i in e.index]
e['exclusion_reason']='user_selected_complete_G1_G6_cases'
e=e.sort_values('candidate_key').reset_index(drop=True)
assert len(x)==189 and len(e)==21
assert x.office_type.value_counts().to_dict()=={'直轄市議員':178,'直轄市長':11}
assert e.missing_fields.eq('G1_age_2026_minus_birth_year').all()
assert e.G5_first_time_any_elected_office.eq('1').all()
assert x.G3_unpublished_lowest_flag.eq('1').sum()==5
assert x.loc[x.candidate_name=='游智彬','eligibility_provisional_flag_20261108'].eq('1').all()
assert set(x.candidate_key).isdisjoint(e.candidate_key)
assert set(x.candidate_key)|set(e.candidate_key)==set(d.candidate_key)
pd.testing.assert_frame_equal(x.set_index('candidate_key')[d.columns.drop('candidate_key')].sort_index(),d.set_index('candidate_key').loc[x.candidate_key].sort_index())
save(x,'data/processed/tk_g1_g6_complete_20261111.csv');save(e,'data/processed/tk_g1_g6_exclusions_20261111.csv')
assert sha(p)=='9df3c70602e5ac1293422dfed1a4a008e07d7280b069225d30a3cb250000a8a0'
display(x[['candidate_name','county_city',*COLS]].head(10))
print('189人、三池、21排除、原背景逐格一致與磁碟回讀通過。')


data/processed/tk_g1_g6_complete_20261111.csv 189 59 7683905c256c987844d112d003bdb2595fafe1efa41d6524e211e1db9f6f67eb
data/processed/tk_g1_g6_exclusions_20261111.csv 21 8 3b3140881b051940a79be739993e96affc3f25bcc6f5ec072e7e39467a2592dd


,candidate_name,county_city,G1_age,G2_sex_category,G3_education_category,G4_party_category,G5_first_candidacy,G6_family_search
0,郭璽,臺北市,68,男,大學,其他政黨,0,0
1,沈伯洋,臺北市,44,男,研究所,民進黨,0,1
2,蕭文乾,臺北市,52,男,研究所,其他政黨,0,0
3,唐新民,臺北市,76,男,研究所,其他政黨,0,0
4,蔣萬安,臺北市,48,男,研究所,國民黨,0,1
5,林志成,臺北市,61,男,研究所,無黨籍／未經政黨推薦,0,0
6,張靜,高雄市,70,男,大學,其他政黨,0,0
7,賴瑞隆,高雄市,53,男,研究所,民進黨,0,0
8,王肇民,高雄市,63,男,大學,無黨籍／未經政黨推薦,1,0
9,柯志恩,高雄市,64,女,研究所,國民黨,0,1


189人、三池、21排除、原背景逐格一致與磁碟回讀通過。
